# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* **Unit of Analysis (Grain):** One row represents a single page (`URL`) on a specific date (`date`).
* **Time Window:** Mid-panel development month from March 1, 2026 to March 31, 2026 (`2026-03`), holding back the final month (`2026-06`) as a sealed test set.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Context:**
  * `url`: Unique page descriptor.
  * `date`: Daily observation timestamp.

* **Label:**
  * `target_clicks`: Future daily clicks for the target page.

* **Feature:**
  * `impressions`: Daily search result impressions.
  * `position`: Average daily search ranking position.
  * `ctr`: Daily click-through rate (`clicks / impressions`).
  * `lag_impressions_7d`: Rolling 7-day average of impressions prior to the observation date.
  * `lag_clicks_7d`: Rolling 7-day average of clicks prior to the observation date.

* **Excluded:**
  * Rows with `impressions = 0`: Deliberately excluded to eliminate noise from inactive pages and focus only on active search traffic.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
import duckdb

# تنفيذ الاستعلام والتأكد من عدم وجود تكرار في الـ Grain
query_1 = """
SELECT 
    url, 
    date, 
    COUNT(*) AS duplicate_count
FROM page_daily
WHERE date >= '2026-03-01' AND date <= '2026-03-31'
GROUP BY url, date
HAVING COUNT(*) > 1;
"""

# تشغيل الاستعلام وعرض النتيجة
result_1 = duckdb.sql(query_1).df()
print(f"عدد الصفوف المكررة: {len(result_1)}")
result_1.head()

In [ ]:
# Query 2: Row count and date span for mid-panel month (2026-03)
query_2 = """
SELECT 
    COUNT(*) AS total_rows, 
    MIN(date) AS start_date, 
    MAX(date) AS end_date
FROM page_daily
WHERE date >= '2026-03-01' AND date <= '2026-03-31';
"""
result_2 = duckdb.sql(query_2).df()
result_2

In [ ]:
# Query 3: Availability check for active rows (impressions > 0)
query_3 = """
SELECT 
    COUNT(*) AS active_rows
FROM page_daily
WHERE date >= '2026-03-01' AND date <= '2026-03-31'
  AND impressions > 0;
"""
result_3 = duckdb.sql(query_3).df()
result_3

In [ ]:
# Demonstration of intentional label leakage
# Adding a leaked column derived from the target
df['leaked_feature'] = df['target_clicks'] * 1.0 

# Notice model score jumps to near 1.0 (unrealistic perfection)

# Clean up / Drop the leaked feature to keep the honest baseline
df = df.drop(columns=['leaked_feature'])

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* **Data Limitation:** 
  Search Console data reflects organic search traffic and search performance only. It does not account for direct/referral web traffic, off-site campaigns, or seasonal intent shifts outside the single mid-panel observation window.

## Self-check

Before you submit, confirm each line honestly:

- [ x ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x ] No client names, URLs, or private queries anywhere
- [ x ] My claims use careful words: observed, measured, directional, decision-support
- [ ]x  Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.